# [2단계] 검증 후 재학습 — 본 학습 (Colab · A100)

전체 2단계 학습의 **2단계: 1단계에서 드러난 문제를 고쳐 다시 학습하는 본 학습**.
1단계(`train_colab_1_verify.ipynb`)에서 원본 학습셋으로 파이프라인을 검증한 결과 → **C2·C3 희소, C1 과예측** 확인.
이를 ① 데이터 보강(2차 expert_control로 C2/C3 추가) ② focal loss ③ class weight 완화 ④ LR↓(2e-5→1e-5)로 대응.

**실행 순서:** ① 런타임 → 변경 → GPU(A100) ② 위에서 아래로 셀 실행

**업로드 파일 2개** (`data/labeled/`):
- `tsla_train_enriched_final.csv` — 학습셋 (2차 expert_control로 C2/C3 보강)
- `nvda_eval_final.csv` — 평가셋 (자연분포 유지, 누수 없음 / 1단계와 동일)

**설계 (PLAN Step 4·5)**
- 입력 = 텍스트 + 주주여부(`[주주]`/`[비주주]` prepend). 주가·뱃지·미래정보 미포함.
- KcELECTRA 특수토큰 함정 보정: `TemplateProcessing`으로 `[CLS]`/`[SEP]` 강제.
- 불균형 대응: class weight(완화) + focal loss + macro-F1 기준 early stopping.
- 학습셋 내부에서만 val 분리 → 평가셋(NVDA) 누수 차단.
- **Fold A 단방향**(TSLA학습→NVDA평가): 3모델 벤치마크 → macro-F1 최고 선택 + 베이스라인.
- ※ 비용상 Fold B(NVDA→TSLA)는 생략.

## 0. 환경 준비 & 데이터 업로드

In [ ]:
!pip install -q -U transformers accelerate scikit-learn koreanize-matplotlib
import torch, koreanize_matplotlib
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "런타임 → 변경 → GPU(A100)를 선택하세요."

In [ ]:
from google.colab import files
up = files.upload()   # tsla_train_enriched_final.csv, nvda_eval_final.csv
print("업로드됨:", list(up))

## 1. 설정 (CONFIG)

F1 개선 노브: `MERGE_C1C2`(3-class 병합), `FOCAL_GAMMA`(소수클래스 집중), `WEIGHT_TEMPER`(class weight 완화 → C1 과예측 억제).

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments,
                          EarlyStoppingCallback)

DATA_DIR = "."                  # 업로드한 CSV 위치
FOLDS = {
    "A": dict(train="tsla_train_enriched_final.csv", test="nvda_eval_final.csv",
              note="TSLA학습(보강)→NVDA평가"),
}
# 벤치마크 후보 (ModernBERT-ko 체크포인트는 분류 적합성 확인 후 추가 가능).
MODELS = {
    "KcELECTRA":    "beomi/KcELECTRA-base",   # 주력(댓글 도메인)
    "KLUE-RoBERTa": "klue/roberta-base",      # 한국어 NLU 표준 레퍼런스
    "KR-FinBERT":   "snunlp/KR-FinBert",      # 금융 도메인(ablation 성격)
}

# C1(실패)·C2(적중) 병합 = 3-class. 텍스트로 적중/실패 구분은 본질 불가(미래정보 필요).
MERGE_C1C2 = False
if MERGE_C1C2:
    NUM_LABELS, LABELS, NAMES = 3, [0, 1, 2], ["C0예측없음", "C1+2 예측함", "C3날짜적중"]
else:
    NUM_LABELS, LABELS, NAMES = 4, [0, 1, 2, 3], ["C0예측없음", "C1실패", "C2방향적중", "C3날짜적중"]

MAX_LEN = 512          # PLAN 확정. 짧은 댓글은 dynamic padding으로 효율 처리
VAL_SIZE = 0.15        # 학습셋에서 떼는 검증 비율(early stopping용, 평가셋과 분리)
EPOCHS = 6
BATCH = 32
LR = 1e-5              # 과적합 완화(2e-5→1e-5)
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
FOCAL_GAMMA = 2.0      # focal loss(0이면 일반 CE). 쉬운 다수클래스 비중↓
WEIGHT_TEMPER = 0.5    # class weight 완화 지수(1=원래, 0.5=sqrt) → C1 과예측 억제
SEED = 42
HOLDER_TOKENS = ["[주주]", "[비주주]"]
OUT_DIR = "runs"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"device={DEVICE}  NUM_LABELS={NUM_LABELS}  MERGE_C1C2={MERGE_C1C2}")

## 2. 데이터 로드 & 분포 확인 + 희소클래스 보강 근거

학습셋(보강)·평가셋의 Class 분포를 찍고, **2차 라벨링으로 C2·C3를 얼마나 늘렸는지**(원본 대비)를 시각화한다.

In [ ]:
def make_inputs(df: pd.DataFrame) -> list:
    """텍스트 앞에 [주주]/[비주주] prepend (입력 = 텍스트 + 주주여부)."""
    holder = df["주주_여부"].astype(str).str.lower().eq("true").map(
        {True: "[주주]", False: "[비주주]"})
    return (holder + " " + df["text"].astype(str)).tolist()


def load_fold(fold: str):
    f = FOLDS[fold]
    tr = pd.read_csv(os.path.join(DATA_DIR, f["train"]), encoding="utf-8-sig")
    te = pd.read_csv(os.path.join(DATA_DIR, f["test"]), encoding="utf-8-sig")
    for d in (tr, te):
        d.dropna(subset=["Class", "text"], inplace=True)
        d["Class"] = d["Class"].astype(int)
        if MERGE_C1C2:                       # {0:0, 1:1, 2:1, 3:2}
            d["Class"] = d["Class"].map({0: 0, 1: 1, 2: 1, 3: 2})
    return tr, te, f["note"]


trA, teA, noteA = load_fold("A")
print("학습셋(TSLA) Class 분포:", dict(trA["Class"].value_counts().sort_index()), "| n =", len(trA))
print("평가셋(NVDA) Class 분포:", dict(teA["Class"].value_counts().sort_index()), "| n =", len(teA))
print("입력 예시:", make_inputs(trA.head(1))[0][:80], "...")

In [ ]:
# =============================================================================
# [보강 근거 시각화] 2차 라벨링(expert_control)으로 희소 클래스 C2·C3 학습표본 보강
#   출처: PROGRESS.md / fig_enrich.py — 본 노트북은 업로드 2파일만 쓰므로 수치는 상수로 기입.
#     · 원본(tsla_train_final)     C0=12029 C1=2319 C2=608 C3=35  (n=14,991)
#     · 보강(tsla_train_enriched)  C0=21503 C1=5566 C2=909 C3=60  (n=28,038)
#     · 적중(C2+C3) 출처: 1차 643 + 2차 고수후보 200 + 2차 대조군 126 = 969
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
import koreanize_matplotlib  # 한글 폰트 (0번 셀에서 설치)

NEUTRAL, UP, DOWN = "#9E9E9E", "#E31A1C", "#1F6FE5"   # 원본 / 추가분(고수후보) / 추가분(대조군)
CLASS_LBL = ["C0\n예측없음", "C1\n실패", "C2\n방향적중", "C3\n날짜적중"]
orig = np.array([12029, 2319, 608, 35])
enr  = np.array([21503, 5566, 909, 60])
S1, Sg, Sc = 643, 200, 126           # 적중 출처: 1차 / 2차 고수후보 / 2차 대조군
tot = S1 + Sg + Sc                    # = 969

fig, (axL, axR) = plt.subplots(1, 2, figsize=(14, 5.8),
                               gridspec_kw={"width_ratios": [1.5, 1]})

# ── ① 클래스별 건수: 원본 vs 보강 (log y) ──
x, w = np.arange(4), 0.38
axL.bar(x - w / 2, orig, w, color=NEUTRAL, label=f"원본 (n={orig.sum():,})")
axL.bar(x + w / 2, enr,  w, color=UP,      label=f"보강 후 (n={enr.sum():,})")
axL.set_yscale("log")
axL.set_xticks(x); axL.set_xticklabels(CLASS_LBL, fontsize=10.5)
axL.set_xlabel("클래스  (예측 없음 → 실패 → 방향 적중 → 날짜 적중)", fontsize=11)
axL.set_ylabel("댓글 수  —  로그 스케일", fontsize=11)
axL.set_title("① 클래스별 학습 표본 수 — 희소한 C2·C3가 얼마나 늘었나",
              fontsize=13, fontweight="bold", pad=8)
for xi, (a, b) in enumerate(zip(orig, enr)):
    axL.annotate(f"{a:,}", (xi - w / 2, a), ha="center", va="bottom", fontsize=8.5, color="0.35")
    axL.annotate(f"{b:,}\n(+{(b - a) / a * 100:.0f}%)", (xi + w / 2, b), ha="center", va="bottom",
                 fontsize=8.5, color=UP, fontweight="bold")
axL.legend(fontsize=10.5, loc="upper right", framealpha=0.9)
axL.margins(y=0.22)

# ── ② 적중(C2+C3) 표본의 출처 ──
axR.bar(0, S1, 0.5, color=NEUTRAL, label=f"1차 라벨링   {S1}")
axR.bar(0, Sg, 0.5, bottom=S1,        color=UP,   label=f"2차 고수후보  +{Sg}")
axR.bar(0, Sc, 0.5, bottom=S1 + Sg,   color=DOWN, label=f"2차 대조군    +{Sc}")
for y0, v in [(S1 / 2, S1), (S1 + Sg / 2, Sg), (S1 + Sg + Sc / 2, Sc)]:
    axR.annotate(f"{v}", (0, y0), ha="center", va="center", fontsize=12, fontweight="bold", color="white")
axR.annotate(f"총 {tot}건\n(+{(tot - S1) / S1 * 100:.0f}%)", (0, tot), ha="center", va="bottom",
             fontsize=12, fontweight="bold")
axR.set_xlim(-0.7, 0.7); axR.set_xticks([])
axR.set_ylabel("적중 댓글 수  (C2 + C3)", fontsize=11)
axR.set_title("② 적중 표본은 어디서 왔나 — 643 → 969건",
              fontsize=13, fontweight="bold", pad=8)
axR.legend(fontsize=10.5, loc="upper left", framealpha=0.9)
axR.margins(y=0.16)

fig.suptitle("2차 라벨링으로 희소 클래스(C2·C3) 학습 표본 보강 — TSLA 학습셋",
             fontsize=15.5, fontweight="bold", y=1.02)
fig.text(0.5, 0.945,
         "전체 분포 비중은 ~3.5%로 유지(C0도 함께 증가) · 핵심은 적중 표본의 '절대량' 증가 → 러닝커브 데이터 부족 완화",
         ha="center", fontsize=10, color="0.45")
plt.tight_layout(rect=[0, 0, 1, 0.93]); plt.show()

print(f"원본  Class: C0={orig[0]:,} C1={orig[1]:,} C2={orig[2]} C3={orig[3]}  (n={orig.sum():,})")
print(f"보강  Class: C0={enr[0]:,} C1={enr[1]:,} C2={enr[2]} C3={enr[3]}  (n={enr.sum():,})")
print(f"적중(C2+C3) 출처: 1차 {S1} / 2차 고수후보 {Sg} / 2차 대조군 {Sc} = {tot}건")

## 3. 토크나이저 / 데이터셋 / Trainer 유틸

KcELECTRA 등 `post_processor`가 없는 토크나이저는 `[CLS]`/`[SEP]`가 자동 추가되지 않으므로 `TemplateProcessing`으로 강제 보정.

In [ ]:
class TextDS(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tok):
        self.enc = tok(texts, truncation=True, max_length=MAX_LEN)
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: self.enc[k][i] for k in self.enc}
        item["labels"] = int(self.labels[i])
        return item


def build_tokenizer(model_name: str):
    tok = AutoTokenizer.from_pretrained(model_name)
    tok.add_special_tokens({"additional_special_tokens": HOLDER_TOKENS})
    # post_processor 없는 토크나이저: [CLS]/[SEP] 자동추가 안 됨 → 강제
    ids = tok("테스트")["input_ids"]
    if tok.cls_token_id is not None and (len(ids) == 0 or ids[0] != tok.cls_token_id):
        from tokenizers.processors import TemplateProcessing
        tok._tokenizer.post_processor = TemplateProcessing(
            single=f"{tok.cls_token} $A {tok.sep_token}",
            pair=f"{tok.cls_token} $A {tok.sep_token} $B:1 {tok.sep_token}:1",
            special_tokens=[(tok.cls_token, tok.cls_token_id),
                            (tok.sep_token, tok.sep_token_id)])
        chk = tok("테스트")["input_ids"]
        assert chk[0] == tok.cls_token_id and chk[-1] == tok.sep_token_id, "CLS/SEP 보정 실패"
        print(f"    [특수토큰 보정] {model_name}: [CLS]/[SEP] TemplateProcessing 적용")
    return tok


class WeightedTrainer(Trainer):
    def __init__(self, *a, class_weights=None, **k):
        super().__init__(*a, **k)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        w = self.class_weights.to(out.logits.device)
        ce = F.cross_entropy(out.logits, labels, weight=w, reduction="none")
        if FOCAL_GAMMA > 0:                              # focal: 쉬운 샘플 비중↓
            pt = torch.exp(-F.cross_entropy(out.logits, labels, reduction="none"))
            loss = ((1 - pt) ** FOCAL_GAMMA * ce).mean()
        else:
            loss = ce.mean()
        return (loss, out) if return_outputs else loss


def metrics_fn(p):
    preds = p.predictions.argmax(-1)
    return {"macro_f1": f1_score(p.label_ids, preds, average="macro"),
            "acc": accuracy_score(p.label_ids, preds)}


def report(name, y_true, y_pred):
    mf1 = f1_score(y_true, y_pred, average="macro")
    print(f"\n  ── {name} ──  macro-F1 = {mf1:.4f}  acc = {accuracy_score(y_true, y_pred):.4f}")
    print(classification_report(y_true, y_pred, labels=LABELS, target_names=NAMES,
                                digits=3, zero_division=0))
    print("  혼동행렬 (행=실제, 열=예측):\n", confusion_matrix(y_true, y_pred, labels=LABELS))
    return mf1

## 4. 단일 모델 fine-tuning + 교차평가 함수

In [ ]:
def run_model(model_name, hf_id, tr_df, te_df, note):
    tok = build_tokenizer(hf_id)
    Xtr = make_inputs(tr_df); ytr = tr_df["Class"].to_numpy()
    Xte = make_inputs(te_df); yte = te_df["Class"].to_numpy()
    # 학습셋 내부 train/val(early stopping용) — 평가셋(타 종목)은 손대지 않음
    Xt, Xv, yt, yv = train_test_split(Xtr, ytr, test_size=VAL_SIZE,
                                      stratify=ytr, random_state=SEED)
    ds_tr, ds_va, ds_te = TextDS(Xt, yt, tok), TextDS(Xv, yv, tok), TextDS(Xte, yte, tok)

    model = AutoModelForSequenceClassification.from_pretrained(hf_id, num_labels=NUM_LABELS)
    model.resize_token_embeddings(len(tok))     # [주주]/[비주주] 추가분 반영

    cw = compute_class_weight("balanced", classes=np.arange(NUM_LABELS), y=yt)
    cw = torch.tensor(cw, dtype=torch.float) ** WEIGHT_TEMPER   # 가중 완화(C1 과예측 억제)

    args = TrainingArguments(
        output_dir=os.path.join(OUT_DIR, f"A_{model_name}"),
        num_train_epochs=EPOCHS, learning_rate=LR,
        weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=64,
        eval_strategy="epoch", save_strategy="epoch", logging_steps=50,
        load_best_model_at_end=True, metric_for_best_model="macro_f1",
        greater_is_better=True, save_total_limit=1, seed=SEED, report_to="none",
        fp16=torch.cuda.is_available())
    trainer = WeightedTrainer(
        model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
        data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics_fn,
        class_weights=cw, callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
    trainer.train()
    pred = trainer.predict(ds_te).predictions.argmax(-1)
    return report(f"[A] {model_name} ({note})", yte, pred)

## 5. 벤치마크 — Fold A (TSLA학습 → NVDA평가)

3모델 비교 → macro-F1 최고 모델 선택. (A100 기준 모델당 수 분 소요)

In [ ]:
scores = {}
for name, hf_id in MODELS.items():
    print(f"\n===== {name} =====")
    try:
        scores[name] = run_model(name, hf_id, trA, teA, noteA)
    except Exception as e:
        print(f"  ⚠️ {name} 실패: {e}")

best = max(scores, key=scores.get)
print("\n벤치마크 macro-F1:", {k: round(v, 4) for k, v in scores.items()})
print(f"→ 최고 모델: {best} (macro-F1 {scores[best]:.4f})")

## 6. 베이스라인 (다수클래스 / TF-IDF + LogReg)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

Xtr, ytr = make_inputs(trA), trA["Class"].to_numpy()
Xte, yte = make_inputs(teA), teA["Class"].to_numpy()

# 다수 클래스
maj = np.bincount(ytr, minlength=NUM_LABELS).argmax()
bl_maj = report(f"[A] 베이스라인-다수클래스(={maj})", yte, np.full_like(yte, maj))

# TF-IDF + LogReg
vec = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2)
Ztr, Zte = vec.fit_transform(Xtr), vec.transform(Xte)
lr = LogisticRegression(max_iter=1000, class_weight="balanced", n_jobs=-1)
lr.fit(Ztr, ytr)
bl_tfidf = report("[A] 베이스라인-TFIDF+LogReg", yte, lr.predict(Zte))

## 7. 결과 요약 (macro-F1 비교)

In [ ]:
import matplotlib.pyplot as plt

summary = {"다수클래스": bl_maj, "TFIDF+LogReg": bl_tfidf, **scores}
summary = dict(sorted(summary.items(), key=lambda kv: kv[1]))

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#bbb" if k in ("다수클래스", "TFIDF+LogReg") else "#4C78A8" for k in summary]
bars = ax.barh(list(summary), list(summary.values()), color=colors)
ax.bar_label(bars, fmt="%.3f", padding=3)
ax.set_xlabel("macro-F1 (NVDA 평가셋)")
ax.set_title("Fold A: TSLA학습 → NVDA평가 — 모델/베이스라인 비교")
ax.set_xlim(0, max(summary.values()) * 1.15)
plt.tight_layout(); plt.show()

print(f"\n최고 모델: {best} (macro-F1 {scores[best]:.4f})")
print("※ 단방향(TSLA→NVDA) 일반화 결과만 보고. Fold B(NVDA→TSLA)는 비용상 생략.")